# Volatility Environment Phase 3 — Risk Allocation Economic Value

Retrospective analysis, not new OOS. Fixed 28 strategies; no live adoption. Plan SHA: a7f55489a36ebe353f74e4571222ae16ae0615cd.

R0=.90% throughout; R1=.70/.80/.90/1.00/1.10%; R2=.50/.70/.90/1.10/1.30%; F1 skips Q1 and uses .90% otherwise. Insufficient history=.90% for every candidate.

Results pending the preregistered run.

In [ ]:
# Colab: run this setup only when regenerating from original inputs.
from pathlib import Path
import subprocess, sys, json
REPO_URL='https://github.com/TR-KJ/time-entry-portfolio-lab.git'
IMPLEMENTATION_SHA=''  # populated in the result notebook; implementation cannot embed its own SHA
ROOT=Path('/content/volatility_phase3_repo')
OUT=Path('/content/volatility_phase3')
if not IMPLEMENTATION_SHA:
    raise ValueError('Use the result notebook containing the verified implementation SHA')
if not ROOT.exists():
    subprocess.run(['git','clone',REPO_URL,str(ROOT)],check=True)
subprocess.run(['git','-C',str(ROOT),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach',IMPLEMENTATION_SHA],check=True)


In [ ]:
# Set uploaded original input paths. No Drive mounting is required.
BASELINE=Path('/content/daily_stop_baseline_trades.csv')
M1_ROOT=Path('/content/m1')
INPUT_PATHS=Path('/content/volatility_phase3_input_paths.json')
INPUT_PATHS.write_text(json.dumps([str(p) for p in M1_ROOT.rglob('*.csv')]))
env=__import__('os').environ.copy()
env['PYTHONPATH']=str(ROOT/'src/research')+':'+str(ROOT/'tests')
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-p','test_volatility_phase*.py'],env=env,check=True)
subprocess.run([sys.executable,str(ROOT/'src/research/volatility_phase3.py'),'--baseline',str(BASELINE),'--input-paths',str(INPUT_PATHS),'--output-dir',str(OUT),'--implementation-sha',IMPLEMENTATION_SHA],env=env,check=True)


In [ ]:
import pandas as pd
from IPython.display import display
for name in ['money_summary','period_reset_summary','risk_allocation_summary','robustness_summary','decision','verification']:
    print(name)
    display(pd.read_csv(OUT/f'volatility_phase3_{name}.csv'))
# Verify serialized outputs against this execution's run record.
import hashlib
record=pd.read_csv(OUT/'volatility_phase3_run_record.csv').iloc[0]
for name,expected in json.loads(record.OutputHashes).items():
    assert hashlib.sha256((OUT/name).read_bytes()).hexdigest()==expected, name


In [ ]:
SAVE_TO_DRIVE=False
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    shutil.copytree(OUT,'/content/drive/MyDrive/volatility_phase3',dirs_exist_ok=True)
